# 02 — Baseline Training: EfficientNet-B3 on CIFAKE

This notebook:
1. Sets seeds and detects GPU
2. Builds the EfficientNet-B3 detector
3. Configures optimizer, scheduler, and loss
4. Runs 20-epoch training loop with checkpointing
5. Plots training curves
6. Evaluates on the test set (accuracy, AUC, ECE)

**Prerequisites:** Run `01_setup_and_eda.ipynb` first to download CIFAKE.

In [ ]:
# Cell 1 — Imports, seeds, device
import os, sys
import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm

REPO_DIR = "/content/ai-image-detection"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

DRIVE_ROOT = "/content/drive/MyDrive/ai-image-detection"
DATA_DIR = os.path.join(REPO_DIR, "data", "raw", "cifake")
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
PLOTS_DIR = os.path.join(REPO_DIR, "outputs", "plots")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2 — Build model & load data
from src.model.detector import build_detector
from src.utils.data_loader import get_cifake_loaders

model = build_detector(pretrained=True).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

train_loader, val_loader, test_loader = get_cifake_loaders(
    DATA_DIR, batch_size=32, num_workers=2,
)

In [ ]:
# Cell 3 — Optimizer, scheduler, loss
NUM_EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()

print(f"Optimizer:  AdamW (lr={LR}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler:  CosineAnnealingLR (T_max={NUM_EPOCHS})")
print(f"Loss:       CrossEntropyLoss")
print(f"Epochs:     {NUM_EPOCHS}")

In [ ]:
# Cell 4 — Training loop
from src.model.detector import save_checkpoint

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_detector.pth")


def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    ctx = torch.no_grad() if not training else torch.enable_grad()
    with ctx:
        for images, labels in tqdm(loader, leave=False):
            images, labels = images.to(device), labels.to(device)

            if training:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += images.size(0)

    return running_loss / total, correct / total


for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    lr_now = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch:2d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} | "
        f"LR: {lr_now:.6f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        metrics = {"val_acc": val_acc, "val_loss": val_loss, "train_acc": train_acc}
        save_checkpoint(model, best_checkpoint_path, epoch, metrics)
        print(f"  -> Saved best checkpoint (val_acc={val_acc:.4f})")

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")

In [ ]:
# Cell 5 — Plot training curves
import matplotlib.pyplot as plt

epochs = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(epochs, history["train_loss"], label="Train")
ax1.plot(epochs, history["val_loss"], label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(epochs, history["train_acc"], label="Train")
ax2.plot(epochs, history["val_acc"], label="Val")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy")
ax2.legend()

fig.suptitle("EfficientNet-B3 Training on CIFAKE", fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "training_curves.png"), dpi=150)
plt.show()
print(f"Saved to {PLOTS_DIR}/training_curves.png")

In [ ]:
# Cell 6 — Test set evaluation
from src.model.detector import load_checkpoint
from src.evaluation.metrics import (
    compute_accuracy, compute_auc, compute_ece,
    plot_reliability_diagram, plot_roc_curve,
)

# Load best checkpoint
epoch_loaded, ckpt_metrics = load_checkpoint(model, best_checkpoint_path)
model.to(device)
model.eval()
print(f"Loaded checkpoint from epoch {epoch_loaded} (val_acc={ckpt_metrics['val_acc']:.4f})")

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)
all_preds = np.argmax(all_probs, axis=1)

acc = compute_accuracy(all_preds, all_labels)
auc = compute_auc(all_probs, all_labels)
ece = compute_ece(all_probs, all_labels)

print(f"\n{'='*40}")
print(f"  Test Results (CIFAKE)")
print(f"{'='*40}")
print(f"  Accuracy : {acc:.4f}")
print(f"  AUC-ROC  : {auc:.4f}")
print(f"  ECE      : {ece:.4f}")
print(f"{'='*40}")

plot_reliability_diagram(all_probs, all_labels, os.path.join(PLOTS_DIR, "reliability_diagram.png"))
plot_roc_curve(all_probs, all_labels, os.path.join(PLOTS_DIR, "roc_curve.png"))
print(f"\nPlots saved to {PLOTS_DIR}/")